## Actividad 3_6

<div style="border-style:groove;border-width:thin;padding:10px">

En esta actividad vamos a intentar solucionar un problema de regresión con uno de los métodos que hemos visto en clase hasta ahora:
- Regresión Lineal Simple.
- Regresión Lineal Múltiple.
- Regresión Polinómica.
</div>

<p style="border-style:groove;border-width:thin;padding:10px">
Lo primero que vamos a hacer es importar los datos y analizar el dataset que tenemos.
</p>

In [2]:
import pandas as pd 

df_coches = pd.read_csv("CarPrice_Assignment.csv")
df_coches.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


<p style="border-style:groove;border-width:thin;padding:10px">
A continuación vamos a modificar el dataset para eliminar lo que no nos interesa y cambiar las columnas para poder hacer una regresión.
    
</p>

In [3]:
df_coches_limpio = df_coches.drop(columns=["car_ID"])
df_coches_limpio.head()

,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [4]:
df_coches_limpio["marca"] = df_coches_limpio["CarName"].str.split(" ").str[0]
df_coches_limpio["modelo"] = df_coches_limpio["CarName"].str.split(" ").str[1:].str.join(" ")
df_coches_limpio = df_coches_limpio.drop(columns=["CarName", "modelo"])

df_coches_limpio.head()

,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,marca
0,3,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,...,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,alfa-romero
1,3,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,...,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,alfa-romero
2,1,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,...,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,alfa-romero
3,2,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,...,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0,audi
4,2,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,...,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0,audi


<div style="border-style:groove;border-width:thin;padding:10px">
    Ahora comprobamos si hay valores nulos. A continuación vamos a proceder a cambiar las columnas que tienen categorías ("categorical features") para poder realizar una regresión con ellas. 
<p>Antes que nada, vamos a definir que son columnas categóricas. Son columnas cuyos datos deben pertenecer a un conjunto de valores finito. Este conjunto de valores puede ser numérico (en cuyo caso podemos usarlo directamente en una regresión) o un texto.</p>
<p>Si nos encontramos con columnas con texto, como es nuestro caso, lo más común es asignar valores numéricos a los valores de las columnas. Esta técnica se llama <b>One-hot encoding</b>. El cambio más habitual para poder realizar una regresión sería convertir la columna en varias, una por cada posible valor. Usando como ejemplo nuestro dataset, la columna <b>fueltype</b> se transformaría en 2 columnas, <b>fueltype-gas y fueltype-diesel</b>.</p>
    <p>Los posibles valores de estas columnas dependeran de la codificación que usemos:</p>
    <ul>
        <li><b>Dummy encoding:</b> Tendrán 0 o 1. En nuestro caso de ejemplo, un coche diesel tendrá 0 en fueltype-gas y 1 en fueltype-diesel.</li>
        <li><b>Simple effect encoding:</b> En vez de 0 y 1 tendrán -0,25 y 0,75. En el mismo ejemplo, el coche diesel tendría -0,25 en fueltype-gas y 0,75 en fueltype-diesel.</li>
    </ul>    
    <p>La función <b>get_dummies</b> de pandas nos permite hacer este cambio en una columa o una lista de columnas. Vamos a hacer un ejemplo con una columna y, después, a moficiar las demás.</p>
</div>

In [12]:
df_coches_limpio.isnull().values.any()

np.False_

In [14]:
df_coches_limpio["doornumber"].replace({"two":2, "four":4}, inplace=True)

/tmp/ipykernel_12264/1775525219.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_coches_limpio["doornumber"].replace({"two":2, "four":4}, inplace=True)
/tmp/ipykernel_12264/1775525219.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_coches_limpio["doornumber"].replace({"two":2, "four":4}

In [19]:
#print(df_coches_limpio.dtypes)
cat_cols = df_coches_limpio.select_dtypes(include="object").columns
num_cols = df_coches_limpio.select_dtypes(exclude="object").columns

print(cat_cols)
print(num_cols)

Index(['fueltype', 'aspiration', 'carbody', 'drivewheel', 'enginelocation',
       'enginetype', 'cylindernumber', 'fuelsystem', 'marca'],
      dtype='object')
Index(['symboling', 'doornumber', 'wheelbase', 'carlength', 'carwidth',
       'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price'],
      dtype='object')


In [16]:
df_modelo = pd.get_dummies(df_coches_limpio, columns=cat_cols, drop_first=True, dtype=int)
df_modelo.head()

,symboling,doornumber,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,...,marca_porsche,marca_renault,marca_saab,marca_subaru,marca_toyota,marca_toyouta,marca_vokswagen,marca_volkswagen,marca_volvo,marca_vw
0,3,2,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,...,0,0,0,0,0,0,0,0,0,0
1,3,2,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,...,0,0,0,0,0,0,0,0,0,0
2,1,2,94.5,171.2,65.5,52.4,2823,152,2.68,3.47,...,0,0,0,0,0,0,0,0,0,0
3,2,4,99.8,176.6,66.2,54.3,2337,109,3.19,3.40,...,0,0,0,0,0,0,0,0,0,0
4,2,4,99.4,176.6,66.4,54.3,2824,136,3.19,3.40,...,0,0,0,0,0,0,0,0,0,0


In [17]:
print(df_modelo.columns)
print(df_modelo.shape)
print(df_modelo.info())

Index(['symboling', 'doornumber', 'wheelbase', 'carlength', 'carwidth',
       'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price', 'fueltype_gas', 'aspiration_turbo', 'carbody_hardtop',
       'carbody_hatchback', 'carbody_sedan', 'carbody_wagon', 'drivewheel_fwd',
       'drivewheel_rwd', 'enginelocation_rear', 'enginetype_dohcv',
       'enginetype_l', 'enginetype_ohc', 'enginetype_ohcf', 'enginetype_ohcv',
       'enginetype_rotor', 'cylindernumber_five', 'cylindernumber_four',
       'cylindernumber_six', 'cylindernumber_three', 'cylindernumber_twelve',
       'cylindernumber_two', 'fuelsystem_2bbl', 'fuelsystem_4bbl',
       'fuelsystem_idi', 'fuelsystem_mfi', 'fuelsystem_mpfi',
       'fuelsystem_spdi', 'fuelsystem_spfi', 'marca_alfa-romero', 'marca_audi',
       'marca_bmw', 'marca_buick', 'marca_chevrolet', 'marca_dodge',
       'marca_honda', 'marca_isuzu', 'marca_jaguar', 

In [20]:
df_modelo.corr(numeric_only=True)['price'].abs().sort_values(ascending=False)[1:]

enginesize             0.874145
curbweight             0.835305
horsepower             0.808139
carwidth               0.759325
cylindernumber_four    0.697762
                         ...   
fuelsystem_4bbl        0.017306
enginetype_ohcf        0.016285
enginetype_rotor       0.004544
cylindernumber_two     0.004544
fuelsystem_mfi         0.002747
Name: price, Length: 70, dtype: float64

<div style="border-style:groove;border-width:thin;padding:10px">
    Hay una columna categórica que tiene valores numéricos codificados en texto. En este caso he optado por modificarla y pasarla a un tipo de dato numérico aunque se podría hacer lo mismo que con las demás.
</div>

<div style="border-style:groove;border-width:thin;padding:10px">
    También se pueden modificar muchas columnas a la vez con el parámetro de get_dummies columns. En este caso no hay que especificar un prefix para que de nombre a las columnas. Pone por defecto el nombre de la columna original.
</div>

<div style="border-style:groove;border-width:thin;padding:10px">
Para generar los conjuntos X e y vamos a eliminar price en X para coger solo esa columna en y.
</div>

In [21]:
X = df_modelo.drop(['price'],axis=1)
y = df_modelo['price'].to_frame()
print(X)

     symboling  doornumber  wheelbase  carlength  carwidth  carheight  \
0            3           2       88.6      168.8      64.1       48.8   
1            3           2       88.6      168.8      64.1       48.8   
2            1           2       94.5      171.2      65.5       52.4   
3            2           4       99.8      176.6      66.2       54.3   
4            2           4       99.4      176.6      66.4       54.3   
..         ...         ...        ...        ...       ...        ...   
200         -1           4      109.1      188.8      68.9       55.5   
201         -1           4      109.1      188.8      68.8       55.5   
202         -1           4      109.1      188.8      68.9       55.5   
203         -1           4      109.1      188.8      68.9       55.5   
204         -1           4      109.1      188.8      68.9       55.5   

     curbweight  enginesize  boreratio  stroke  ...  marca_porsche  \
0          2548         130       3.47    2.68  ...  

<div style="border-style:groove;border-width:thin;padding:10px">
Ahora vamos a entrenar el sistema usando un modelo de regresión lineal. ¿Será suficiente? Vamos a usar todas las columnas. También se podría probar a usar un subconjunto de columnas.
</div>

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [23]:
from sklearn import metrics
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)

print('Raiz del error absoluto medio: ', MAE(y_test,y_pred))
from sklearn.metrics import r2_score
print('R cuadrado: ', r2_score(y_test, y_pred))

Raiz del error absoluto medio:  1877.884727920393
R cuadrado:  0.8874556036341955


<div style="border-style:groove;border-width:thin;padding:10px">
El resultado obtenido es 0.87 de R². Está bastante bien. El error cuadrático medio que estamos teniendo en el conjunto de test es de 3100€. Teniendo en cuenta el precio de un coche no parece un error pequeño. Vamos a intentar hacerlo mejor. 
    <p>Si habéis pintado las relaciones entre las distintas columnas habréis visto que hay algunas que parecen tener una relación polinómica con el precio. En concreto de grado 2. Vamos a probar con una regresión polinómica, de nuevo, con todas las columnas.</p>
</div>

In [ ]:
# Probamos con una regresion polinomica
from sklearn.preprocessing import PolynomialFeatures
poly_features = PolynomialFeatures(degree=2,include_bias=False)
X_poly = poly_features.fit_transform(X)
y = df_modelo['price']


X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size = 0.2, random_state = 0)

from sklearn import metrics
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)

print('Raiz del error cuadrático medio: ', MAE(y_test,y_pred))
from sklearn.metrics import r2_score
print('R cuadrado: ', r2_score(y_test, y_pred))
print('R cuadrado entrenamiento: ', r2_score(y_train, lm.predict(X_train)))

Raiz del error cuadrático medio:  117100.89104594747
R cuadrado:  -784.5005656510656
R cuadrado entrenamiento:  0.9985785355495101


<div style="border-style:groove;border-width:thin;padding:10px">
¿Porqué sale tan mal? ¿Es dummy coding oportuno para lo que estamos haciendo?
El dummy encoding da valores de 0 y 1. El problema de estos valores es que 0 y 1 al cuadrado, al cubo, etc. no cambian de valor, luego hacer la regresión polinómica no tiene mucho sentido.
Vamos a probar con un "Simple effect encoding". Las columnas serán iguales pero en vez de valores de 0 y 1 tendremos -0.25 y 0.75. Para lograr esto simplemente restamos 0.25 a todas las columnas que hemos creado:
</div>

<div style="border-style:groove;border-width:thin;padding:10px">
Vamos a volver a hacer la regresión polinómica. A ver si esta vez obtenemos un resultado mejor que con la regresión lineal. La intuición en este caso nos indica que deberíamos obtener un resultado mejor ya que hay algunas variables que tienen relación de grado 2 con el precio.
</div>

<div style="border-style:groove;border-width:thin;padding:10px">
Esta vez el R² es muy bueno y nos estamos equivocando de media 474€ en cada coche, lo cual es un dato de error bastante bueno. 
</div>